In [1]:
!pip install delta-spark==3.2.0

In [15]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, input_file_name, to_timestamp, to_date, date_sub, current_date, row_number
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from datetime import datetime, timedelta

if "PYSPARK_SUBMIT_ARGS" in os.environ:
    del os.environ["PYSPARK_SUBMIT_ARGS"]

packages = "io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.postgresql:postgresql:42.6.0"
spark = SparkSession.builder \
    .appName("DailyTransactionPipeline") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", packages) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10485760") \
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "matrix") \
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

Spark Version: 3.5.2


In [44]:
df_all = spark.read.csv("s3a://bronze/landing/atm", header=True, inferSchema=True)
max_date = df_all.select(F.max("tx_date")).collect()[0][0]
df_atm = df_all.filter(F.col("tx_date") == max_date) \
    .withColumn("channel", F.lit("ATM")) \
    .withColumn("source_file", F.input_file_name())
df_atm.show(5)

+------------+-------------------+----------+----+------+--------+-------+--------+-----------+----------+-------+--------------------+
|       tx_id|              tx_ts|   card_no| mcc|amount|currency|country|    city|terminal_id|   tx_date|channel|         source_file|
+------------+-------------------+----------+----+------+--------+-------+--------+-----------+----------+-------+--------------------+
|TX0698572469|2026-02-20 13:05:00|CARD300077|6011|  -5.0|     GBP|    AZE|    Baku|   ATM65006|2026-02-20|    ATM|s3a://bronze/land...|
|TX3189450479|2026-02-20 05:30:00|CARD300147|6011|673.05|     TRY|    AZE|   Ganja|   ATM14262|2026-02-20|    ATM|s3a://bronze/land...|
|TX7971451951|2026-02-20 22:57:00|CARD300144|6011|724.37|     AZN|    DEU|  Berlin|   ATM62840|2026-02-20|    ATM|s3a://bronze/land...|
|TX2113712998|2026-02-20 09:37:00|CARD300081|6011|650.26|     TRY|    USA|New York|   ATM35075|2026-02-20|    ATM|s3a://bronze/land...|
|TX8523830552|2026-02-20 17:13:00|CARD300021|601

In [64]:
df_all = spark.read.csv("s3a://bronze/landing/ecom", header=True, inferSchema=True)
max_date = df_all.select(F.max("tx_date")).collect()[0][0]
df_ecom = df_all.filter(F.col("tx_date") == max_date) \
    .withColumn("channel", F.lit("ECOM")) \
    .withColumn("source_file", F.input_file_name())
df_ecom.show(5)

+------------+-------------------+----------+-----------+----+------+--------+-------+--------+----------+-------+--------------------+
|       tx_id|              tx_ts|   card_no|customer_no| mcc|amount|currency|country|    city|   tx_date|channel|         source_file|
+------------+-------------------+----------+-----------+----+------+--------+-------+--------+----------+-------+--------------------+
|        NULL|2026-02-20 03:05:00|CARD300094| CUST100073|4829|481.32|     TRY|   AZE |   Ganja|2026-02-20|   ECOM|s3a://bronze/land...|
|TX2009750632|2026-02-20 10:04:00|CARD300065|       NULL|7995|440.94|     GBP|    USA|   Miami|2026-02-20|   ECOM|s3a://bronze/land...|
|TX4203529010|2026-02-20 13:00:00|CARD300044| CUST100037|4111| 20.66|     AZN|    GEO| Tbilisi|2026-02-20|   ECOM|s3a://bronze/land...|
|TX0229410673|2026-02-20 14:40:00|CARD300148| CUST100114|6011| 49.44|     AZN|   DEU |  Berlin|2026-02-20|   ECOM|s3a://bronze/land...|
|TX9974847299|2026-02-20 18:51:00|CARD300077|   

In [52]:
df_all = spark.read.json("s3a://bronze/landing/pos")
max_date = df_all.select(F.max("tx_date")).collect()[0][0]
df_ecom = df_all.filter(F.col("tx_date") == max_date) \
        .withColumn("channel", F.lit("POS")) \
        .withColumn("source_file", F.input_file_name())
df_pos.show(5)

+------+----------+--------+-------------------+--------------------+----+------------+----------+-------+--------------------+
|amount|   card_no|currency|         event_time|            location| mcc|       tx_id|   tx_date|channel|         source_file|
+------+----------+--------+-------------------+--------------------+----+------------+----------+-------+--------------------+
|109.43|CARD300026|     AZN|2026-02-20 15:11:00|{Sumqayit, AZE, T...|5411|        NULL|2026-02-20|    POS|s3a://bronze/land...|
|145.87|CARD300069|     TRY|2026-02-20 12:21:00|{Sumqayit, AZE, T...|7011|TX3989232788|2026-02-20|    POS|s3a://bronze/land...|
| 23.43|CARD300083|     USD|2026-02-20 10:23:00|{Baku, AZE, T832108}|6011|TX3673638609|2026-02-20|    POS|s3a://bronze/land...|
| 54.54|CARD300114|     AZN|2026-02-20 01:47:00|{Miami, USA, T629...|6300|TX2422797582|2026-02-20|    POS|s3a://bronze/land...|
|103.31|CARD300155|     AZN|2026-02-20 12:13:00|{Berlin, DEU, T83...|5812|TX6964079195|2026-02-20|    PO

In [60]:
df_atm.printSchema()

root
 |-- tx_id: string (nullable = true)
 |-- tx_ts: timestamp (nullable = true)
 |-- card_no: string (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- tx_date: date (nullable = true)
 |-- channel: string (nullable = false)
 |-- source_file: string (nullable = false)



In [65]:
df_ecom.printSchema()

root
 |-- tx_id: string (nullable = true)
 |-- tx_ts: timestamp (nullable = true)
 |-- card_no: string (nullable = true)
 |-- customer_no: string (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- tx_date: date (nullable = true)
 |-- channel: string (nullable = false)
 |-- source_file: string (nullable = false)



In [66]:
df_pos.printSchema()

root
 |-- amount: string (nullable = true)
 |-- card_no: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- location: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- terminal_id: string (nullable = true)
 |-- mcc: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- tx_date: date (nullable = true)
 |-- channel: string (nullable = false)
 |-- source_file: string (nullable = false)



In [67]:
df_pos =df_pos.withColumn("city", col("location.city")) \
              .withColumn("country", col("location.country")) \
              .withColumn("terminal_id", col("location.terminal_id")) \
              .drop("location")

In [68]:
df_pos = df_pos.withColumnRenamed("event_time", "tx_ts")

In [69]:
df_pos.printSchema()

root
 |-- amount: string (nullable = true)
 |-- card_no: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- tx_ts: string (nullable = true)
 |-- mcc: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- tx_date: date (nullable = true)
 |-- channel: string (nullable = false)
 |-- source_file: string (nullable = false)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- terminal_id: string (nullable = true)



In [71]:
df_pos = df_pos.withColumn("amount", col("amount").cast("double")) \
    .withColumn("mcc", col("mcc").cast("int")) \
    .withColumn("tx_ts", col("tx_ts").cast("timestamp"))

In [72]:
df_union = df_atm.unionByName(df_ecom, allowMissingColumns=True) \
                 .unionByName(df_pos, allowMissingColumns=True) \
                 .withColumn("ingest_date", F.current_date())

df_union.write.mode("overwrite").partitionBy("ingest_date").parquet("s3a://bronze/transactions")

In [73]:
df_dq = df_union.withColumn("dq_reason", 
    F.concat_ws(";", 
        F.when(F.col("tx_id").isNull(), "tx_id_null"),
        F.when(F.col("tx_ts").isNull(), "tx_ts_null"),
        F.when(F.col("amount") <= 0, "amount_invalid"),
        F.when(F.col("currency").isNull(), "currency_null")
    )
)

df_good = df_dq.filter(F.col("dq_reason") == "").drop("dq_reason")
df_bad = df_dq.filter(F.col("dq_reason") != "")

df_good.write.mode("overwrite").partitionBy("ingest_date").parquet("s3a://bronze/transactions_good")
df_bad.write.mode("overwrite").partitionBy("ingest_date").parquet("s3a://bronze/transactions_bad")

In [75]:
windowSpec = Window.partitionBy("tx_id").orderBy(F.col("tx_ts").desc())

df_silver = df_good.withColumn("row_num", F.row_number().over(windowSpec)) \
                       .filter(F.col("row_num") == 1) \
                       .drop("row_num")

df_silver.write.mode("overwrite").partitionBy("tx_date").parquet("s3a://silver/transactions")

In [77]:
df_silver.show(5)

+------------+-------------------+----------+----+-------+--------+-------+--------+-----------+----------+-------+--------------------+-----------+-----------+
|       tx_id|              tx_ts|   card_no| mcc| amount|currency|country|    city|terminal_id|   tx_date|channel|         source_file|customer_no|ingest_date|
+------------+-------------------+----------+----+-------+--------+-------+--------+-----------+----------+-------+--------------------+-----------+-----------+
|TX0008465429|2026-02-20 02:32:00|CARD300116|4829|1082.42|     EUR|    USA|New York|       NULL|2026-02-20|   ECOM|s3a://bronze/land...| CUST100090| 2026-02-24|
|TX0010439369|2026-02-20 21:23:00|CARD300058|5732| 127.04|     GBP|    AZE|Sumqayit|       NULL|2026-02-20|   ECOM|s3a://bronze/land...| CUST100047| 2026-02-24|
|TX0013801510|2026-02-20 18:27:00|CARD300139|6300|  44.23|     AZN|    GEO| Tbilisi|    T347608|2026-02-20|    POS|s3a://bronze/land...|       NULL| 2026-02-24|
|TX0052654111|2026-02-20 16:33:00|

In [ ]:
jdbc_url = "jdbc:postgresql://localhost:5432/test"
connection_properties = {
    "user": "matrix",
    "password": "matrix123",
    "driver": "org.postgresql.Driver"
}

tables = ["customers", "cards", "mcc_mapping", "fx_rate"]

for table in tables:
    df_dim = spark.read.jdbc(url=jdbc_url, table=table, properties=connection_properties)
    df_dim.write.format("delta").mode("overwrite").save(f"s3a://bronze/dims/{table}")

df_customers = spark.read.format("delta").load("s3a://bronze/dims/customers")
df_cards = spark.read.format("delta").load("s3a://bronze/dims/cards")
df_mcc = spark.read.format("delta").load("s3a://bronze/dims/mcc_mapping")
df_fx = spark.read.format("delta").load("s3a://bronze/dims/fx_rate")

In [ ]:
df_enriched = df_silver.withColumn("currency", F.upper(F.trim(F.col("currency")))) \
                       .withColumn("country", F.upper(F.trim(F.col("country"))))

df_enriched = df_enriched.join(df_cards.select("card_no", "customer_no"), "card_no", "left") \
                         .withColumn("customer_no", F.coalesce(df_enriched["customer_no"], df_cards["customer_no"])) \
                         .drop(df_cards["customer_no"])

df_enriched = df_enriched.join(df_customers.select("customer_no", "home_country"), "customer_no", "left")

df_enriched = df_enriched.join(F.broadcast(df_mcc), "mcc", "left")

df_enriched = df_enriched.join(df_fx, ["tx_date", "currency"], "left") \
                         .withColumn("amount_azn", F.col("amount") * F.col("rate"))

df_enriched = df_enriched.withColumn("is_foreign", F.col("country") != F.col("home_country")) \
                         .withColumn("is_high_risk", F.col("risk_weight") >= 8)

df_enriched.write.format("delta").mode("overwrite").save("s3a://silver/enriched_transactions")

In [ ]:
df_gold = df_enriched.groupBy("customer_no", "tx_date").agg(
    F.count("tx_id").alias("total_txn_count"),
    F.sum("amount_azn").alias("total_spent_azn"),
    F.sum(F.when(F.col("is_foreign"), 1).otherwise(0)).alias("foreign_txn_count"),
    F.sum(F.when(F.col("is_high_risk"), 1).otherwise(0)).alias("high_risk_txn_count"),
    F.countDistinct("country").alias("distinct_countries_count"),
    F.avg("amount_azn").alias("avg_amount_azn"),
    F.avg("risk_weight").alias("avg_risk_weight")
)

df_gold = df_gold.withColumn("risk_score", 
    (F.col("foreign_txn_count") * 2) + (F.col("high_risk_txn_count") * 3) + F.col("avg_risk_weight")
)

df_gold = df_gold.withColumn("risk_level", 
    F.when(F.col("risk_score") < 10, "LOW") \
     .when((F.col("risk_score") >= 10) & (F.col("risk_score") <= 19), "MEDIUM") \
     .otherwise("HIGH")
)

df_gold_final = df_gold.coalesce(1)

df_gold_final.write.format("delta").mode("overwrite").save("s3a://gold/customer_daily_risk")

df_gold_final.write.jdbc(
    url=jdbc_url, 
    table="public.customer_daily_risk", 
    mode="overwrite", 
    properties={**connection_properties, "batchsize": "10000"}
)

df_gold_final.show()